# Defensive Cybersecurity 50M: first-run pilot

This starts a small from-scratch decoder-only Transformer on one Kaggle GPU. The corpus is intentionally defensive and small; this 100-step run is a pipeline pilot and will not yield broad cybersecurity competence. The private input contains redacted Human-only tone samples.

In [ ]:
import glob, os, shutil, subprocess, sys, torch, tokenizers
if not torch.cuda.is_available(): raise RuntimeError('Kaggle did not provision a GPU; select GPU T4 x2 and retry.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| Tokenizers:', tokenizers.__version__)

In [ ]:
# Locate the attached private input bundle; Kaggle mount paths can vary by dataset slug.
input_root = '/kaggle/input'
print('Kaggle inputs:', os.listdir(input_root))
train_scripts = glob.glob(os.path.join(input_root, '**', 'train_gpt.py'), recursive=True)
if not train_scripts: raise FileNotFoundError('No attached dataset with train_gpt.py found under /kaggle/input')
bundle = os.path.dirname(train_scripts[0])
work = '/kaggle/working/cyber_50m'
data_dir = '/kaggle/working/data'
os.makedirs(work, exist_ok=True); os.makedirs(data_dir, exist_ok=True)
shutil.copy2(os.path.join(bundle, 'train_gpt.py'), os.path.join(work, 'train_gpt.py'))
shutil.copy2(os.path.join(bundle, 'model_config.json'), os.path.join(work, 'model_config.json'))
shutil.copy2(os.path.join(bundle, 'official_defensive_text.jsonl'), os.path.join(data_dir, 'official_defensive_text.jsonl'))
style_path = os.path.join(bundle, 'copilot_style_private.jsonl')
if os.path.exists(style_path): shutil.copy2(style_path, os.path.join(data_dir, 'copilot_style_private.jsonl'))
print('Prepared defensive records:', sum(1 for _ in open(os.path.join(data_dir, 'official_defensive_text.jsonl'), encoding='utf-8')))

In [ ]:
# Start a bounded first run; checkpoints and metrics are written to Kaggle output storage.
subprocess.run([sys.executable, os.path.join(work, 'train_gpt.py'), '--data-dir', data_dir, '--config', os.path.join(work, 'model_config.json'), '--steps', '100', '--batch-size', '2', '--grad-accum', '4', '--out-dir', '/kaggle/working/cyber_50m_pilot'], check=True)

In [ ]:
import json, os
summary_path = '/kaggle/working/cyber_50m_pilot/run_summary.json'
print(open(summary_path, encoding='utf-8').read())
print('Artifacts:', os.listdir('/kaggle/working/cyber_50m_pilot'))